# T6 — Order + number

**Cited only** (X-10; no computation here touches them): causal order
determines the conformal geometry of a distinguishing spacetime
(Hawking, King, McCarthy, J. Math. Phys. 17, 174 (1976); Malament, J. Math.
Phys. 18, 1399 (1977)); the conformal factor is supplied by counting
(Bombelli, Lee, Meyer, Sorkin, PRL 59, 521 (1987)). Overreach named in X-10:
none of this derives spatial dimension from time.

**Computed**: curvature as a count, without irrationals (X-11: a being with
no metric whose "$\pi$ is half"). For a closed polyhedral surface the angle
deficit at a vertex is $1 - (\text{sum of face angles})$ in *turns*; Descartes
(c. 1630) / Gauss–Bonnet: the deficits sum to $\chi$ turns, $\chi = V - E + F$
(Euler 1758); catalog c17 (Regge 1961). On a cube every deficit is $1/4$ turn
and eight of them make 2 turns. Every number below is a `Fraction`.

In [ ]:
import sys, math, json, cmath, random
from fractions import Fraction
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CATALOG.md").exists())
sys.path.insert(0, str(_root / "notebooks"))
from nbkit import ROOT, show_svg, catalog, verify, mutant_must_fail, falsify
import termplot
print("repo root:", ROOT.name)

## Voxel solids: every face angle is a quarter turn

In [ ]:
def surface(voxels):
    # boundary unit squares of a set of unit cubes; each square as a frozenset of 4 vertices
    voxels = set(voxels)
    squares = []
    for (x, y, z) in voxels:
        for axis in range(3):
            for side in (0, 1):
                nb = list((x, y, z)); nb[axis] += 1 if side else -1
                if tuple(nb) in voxels:
                    continue
                base = [x, y, z]; base[axis] += side
                a, b = [i for i in range(3) if i != axis]
                vs = []
                for da, db in ((0, 0), (1, 0), (1, 1), (0, 1)):
                    p = list(base); p[a] += da; p[b] += db
                    vs.append(tuple(p))
                squares.append(vs)
    return squares

def euler_and_deficits(squares, turn=Fraction(1)):
    V = {v for sq in squares for v in sq}
    E = {frozenset((sq[i], sq[(i + 1) % 4])) for sq in squares for i in range(4)}
    chi = len(V) - len(E) + len(squares)
    count = {v: 0 for v in V}
    for sq in squares:
        for v in sq:
            count[v] += 1
    deficits = {v: turn - n * Fraction(1, 4) for v, n in count.items()}
    return chi, deficits

solids = {
    "cube": [(0, 0, 0)],
    "2x1x1 box": [(0, 0, 0), (1, 0, 0)],
    "L-shape": [(0, 0, 0), (1, 0, 0), (0, 1, 0)],
    "3x3x1 ring (torus)": [(i, j, 0) for i in range(3) for j in range(3) if (i, j) != (1, 1)],
}
results = {}
for name, vox in solids.items():
    chi, deficits = euler_and_deficits(surface(vox))
    total = sum(deficits.values())
    hist = {}
    for d in deficits.values():
        hist[d] = hist.get(d, 0) + 1
    results[name] = (chi, total)
    print(f"{name:>20}: chi = {chi:>2}; deficits {dict(sorted(hist.items()))} -> sum = {total} turn(s)")

## Regular polyhedra: rational deficits, no $\pi$

In [ ]:
# (vertices, faces-per-vertex, corner angle of the face in turns): a regular p-gon has corner (p-2)/(2p)
regular = {"tetrahedron": (4, 3, Fraction(1, 6)), "cube": (8, 3, Fraction(1, 4)),
           "octahedron": (6, 4, Fraction(1, 6)), "dodecahedron": (20, 3, Fraction(3, 10)),
           "icosahedron": (12, 5, Fraction(1, 6))}
for name, (V, n, corner) in regular.items():
    d = 1 - n * corner
    print(f"{name:>12}: {V} vertices x deficit {d} = {V * d} turns")

In [ ]:
def check(turn=Fraction(1), expected_total=None):
    ok = True
    for name, vox in solids.items():
        chi, deficits = euler_and_deficits(surface(vox), turn)
        ok &= sum(deficits.values()) == (chi if expected_total is None else expected_total)
    for name, (V, n, c) in regular.items():
        ok &= V * (turn - n * c) == (2 if expected_total is None else expected_total)
    return ok

falsify(check, {"full-turn-is-three-quarters": lambda: {"turn": Fraction(3, 4)},
                "deficits-sum-to-one-turn": lambda: {"expected_total": 1}})

The torus row is the point: its deficits are $+1/4$ at eight convex corners
and $-1/4$ at eight saddle corners, summing to $0 = \chi(T^2)$. Negative
curvature as a count, no ambient space needed (Gauss; X-10's "embedding"
baggage item).

In [ ]:
chi, deficits = euler_and_deficits(surface(solids["3x3x1 ring (torus)"]))
neg = sorted(v for v, d in deficits.items() if d < 0)
print("saddle vertices (deficit -1/4):", neg)

## Falsifier: catalog c17 (Descartes / Regge 1961) must fail under its mutant

In [ ]:
rc, _ = catalog("c17_gauss_bonnet_polyhedra")
assert rc == 0
rc, out = catalog("c17_gauss_bonnet_polyhedra", mutant=True)
mutant_must_fail("c17_gauss_bonnet_polyhedra", rc, out)